# **Linopy** in a nutshell

Check installation and availability of solver  

In [ ]:
import linopy
import highspy

# If no error message occurs then it's correctly installed
print(highspy.Highs())

# It should then show ['highs']
print(linopy.available_solvers)

Solve an easy optimizing problem (see https://linopy.readthedocs.io/en/latest/create-a-model.html)    

Minimize: x + 2y

Subject to: 
- x >= 0
- y >= 0
- 3x + 7y >= 10
- 5x + 2y >= 3 

In [ ]:
from linopy import Model

# 1) Initialize the model
m = linopy.Model()

# 2) Add variables
# You can assign a lower and upper bounds for your variables. The default is unbounded.
# The name argument is optional but can be useful for referencing the variables later
x = m.add_variables(lower=0, name="x")
y = m.add_variables(lower=0, name="y") 

# 3) Add Constraints
m.add_constraints(3 * x + 7 * y >= 10)
m.add_constraints(5 * x + 2 * y >= 3);

# 4) Add the objective function
m.add_objective(x + 2 * y)

# 5) Solve the model
status, termination = m.solve(solver_name="highs", output_flag=True)

print(status)       # 'ok'
print(termination)  # 'optimal'

# Some useful information like solution and runtime 
m.solution
# print(x.solution)
# print(y.solution)
# m.solver
# m.solver.report


Now solve an optimizing problem using coordinates (see https://linopy.readthedocs.io/en/latest/create-a-model-with-coordinates.html)

Minimize: sum x_t + 2y_t

Subject to: 
- x >= 0
- y >= 0
- 3x_t + 7y_t >= 10*t
- 5x_t + 2y_t >= 3*t        
forall t whereas t spans all the range from 0 to 10.

In [ ]:
import linopy
import pandas as pd

# 1) Initialize model
m = linopy.Model()

# 2) Add variables with coordinates
time = pd.Index(range(10), name="time")     # <class 'pandas.RangeIndex'>: RangeIndex(start=0, stop=10, step=1, name='time')

x = m.add_variables(
    lower=0,
    coords=[time],
    name="x",
)
y = m.add_variables(
    lower=0, 
    coords=[time], 
    name="y"
)

# 3) Add constraints
factor = pd.Series(time, index=time)

con1 = m.add_constraints(3 * x + 7 * y >= 10 * factor, name="con1")
con2 = m.add_constraints(5 * x + 2 * y >= 3 * factor, name="con2")
# See model
# m
# Output:
# Linopy LP model
# ===============

# Variables:
# ----------
#  * x (time)
#  * y (time)

# Constraints:
# ------------
#  * con1 (time)
#  * con2 (time)

# Status:
# -------
# initialized

# 4) Add objective function
obj = (x + 2 * y).sum()
m.add_objective(obj)

# 5) Solve the model
m.solve(solver_name="highs", output_flag=False)

# To show solution use
# m.solution
# or 
m.solution.to_dataframe().plot(grid=True, ylabel="Optimal Value");

Creating variables

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from linopy import Model

m = Model()

# Set variable type to integer or binary with integer=True or binary=True (without lower and upper bounds). Default is continuous 
# If lower and upper do not have the same dimension names, the arrays are broadcasted, meaning the dimensions are spanned

# Initializing variables with xarray
lower = xr.DataArray([1, 2, 3], dims=["time"])
upper = xr.DataArray([10, 11, 12], dims=["station"])
m.add_variables(lower, upper, name="supply")

# Initializing variables with numpy arrays
lower = np.array([1, 2])
upper = np.array([10, 10])
m.add_variables(lower, upper, dims=["my-dim"])

# Initializing variables with Pandas objects
lower = pd.Series([1, 1])
upper = pd.Series([10, 12])
m.add_variables(lower, upper, dims=["my-dim"])



Creating Constraints <=, ==, >= 

In [3]:
from linopy import Model

m = Model()

x = m.add_variables(name="x")

# (optional) This creates an unassigned constraint, i.e. they aren't added to the model yet
con = 3 * x <= 10 

c = m.add_constraints(con, name="my-constraint") # or m.add_constraints(3 * x <= 10, name="the-same-constraint")

Coordinate Alignment

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

m = linopy.Model()

time = pd.RangeIndex(5, name="time")
x = m.add_variables(lower=0, coords=[time], name="x")

subset_time = pd.RangeIndex(3, name="time")
y = m.add_variables(lower=0, coords=[subset_time], name="y")

# Adding x (5 time steps) and y (3 time steps) gives an expression over all 5 time steps. 
# Where y has no entry (time 3, 4), the coefficient is zero — i.e. y simply drops out of the sum at those positions.
x + y

# The same applies when multiplying by a constant that covers only a subset of coordinates. 
# Missing positions get a coefficient of zero:
factor = xr.DataArray([2, 3, 4], dims=["time"], coords={"time": [0, 1, 2]})
x * factor

# For constraints, missing right-hand-side values are filled with NaN, which tells linopy to skip the constraint at those positions:
rhs = xr.DataArray([10, 20, 30], dims=["time"], coords={"time": [0, 1, 2]})
con = x <= rhs
con

# For more, especially on the join operator go see the documentation

Constraint (unassigned) [time: 5]:
----------------------------------
[0]: +1 x[0] ≤ 10.0
[1]: +1 x[1] ≤ 20.0
[2]: +1 x[2] ≤ 30.0
[3]: +1 x[3] ≤ nan
[4]: +1 x[4] ≤ nan